# TweetAge · 03 Análisis de errores

Este notebook mira **en qué se equivoca** el mejor modelo (`greedy_ent`, F1 0.5937 en validación) y lo compara con la mezcla vigente (F1 0.5869, 0.60021 en Kaggle). Es la sección de *análisis de errores, resultados por clase y limitaciones* del entregable.

**No entrena nada.** Usa las probabilidades que el `02_experimentos` ya dejó guardadas en la caché, así que corre en segundos. Tampoco genera submissions.

### La idea en palabras simples

Cada modelo lee un tweet y reparte 100 % de "probabilidad" entre los 6 rangos de edad; la predicción es el rango con más probabilidad. El modelo final es una **mezcla** de varios modelos pequeños llamados **canales** (uno mira las letras, otro las palabras, otro a quién se menciona, etc.); cada canal "vota" y la mezcla suma los votos con distintos pesos.

Para saber si un modelo funciona se guarda un pedazo del train que el modelo **no vio** (la **validación**, ≈ 33 900 tweets) y se compara lo que predice con la respuesta real. Esa partición se hace dos veces con dos **semillas** distintas (dos sorteos diferentes de qué tweets van a validación): la de *ajuste* (20261108), con la que se eligieron los parámetros, y la de *confirmación* (20261109), que se usa para medir de forma honesta. **Salvo que se diga otra cosa, todo lo de este notebook es en la semilla de confirmación.**

### Glosario mínimo

| Término | Qué significa aquí |
|---|---|
| **accuracy** | % de tweets en los que el modelo acierta el rango. |
| **precision** de una clase | De los tweets que el modelo dijo "son 25-34", qué % realmente lo son. Baja = el modelo "acusa" de más a esa clase. |
| **recall** de una clase | De los tweets que realmente son 25-34, qué % encontró el modelo. Bajo = a esa clase "se le escapan" tweets. |
| **F1** de una clase | Promedio (armónico) de precision y recall: un solo número para la calidad en esa clase. |
| **F1 macro** | Promedio simple del F1 de las 6 clases. Es la métrica de Kaggle. |
| **matriz de confusión** | Tabla real × predicha: cuántos tweets de cada clase real terminaron en cada clase predicha. |
| **margen** | Probabilidad de la 1.ª clase menos la de la 2.ª. Margen pequeño = el modelo duda entre dos clases. |
| **calibración** | Si el modelo dice "70 % seguro", ¿acierta el 70 % de esas veces? |
| **canal** | Cada modelo de lenguaje que alimenta la mezcla (`char6_cs`, `mentions`, `word_seq3_ent5_tt`...). |
| **ablación** | Quitar una pieza del modelo y medir cuánto empeora: lo que pierde es lo que esa pieza aportaba. |

### Estructura

0. Carga y verificación.
1. Resultados por clase: matriz de confusión y errores entre clases vecinas.
2. Errores por tipo de tweet (RT, URL, plantillas, longitud...).
3. Confianza y calibración.
4. Qué cambió entre la mezcla vigente y `greedy_ent`.
5. Ejemplos reales de errores y qué canal "votó" mal.
6. Ablación: cuánto aporta cada canal y cada decisión.
7. Limitaciones y conclusiones.

Las funciones están en `src/tweetage/error_analysis.py`; aquí solo se llaman y se muestran.

## 0. Carga y verificación

**Qué hace:** carga de la caché las probabilidades de validación de los dos modelos en las dos semillas y comprueba que el F1 recalculado es el mismo que reportó el `02`. Si la tabla dice `coincide = False`, la caché no corresponde a los mismos datos o a la misma spec y nada de lo que sigue es confiable.

**Detalle técnico:** los canales de caracteres, forma y entidades se leen de la caché de la variante base (`seed{s}`) y los 4 canales de palabras de la variante con TweetTokenizer (`seed{s}_A2_tweettok`), igual que la sección 14 del 02. Si falta algo en la caché, se entrena (≈ 2 min por semilla) y queda guardado.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tweetage import error_analysis as EA
from tweetage import evaluation as E
from tweetage import paths
from tweetage.pipeline import CLASES

N_JOBS = 2
SEMILLA_EJEMPLOS = 20261109  # solo para sortear qué ejemplos se muestran
AJ, CF = EA.SEED_AJUSTE, EA.SEED_CONFIRMACION
FIG_DIR = paths.work_dir() / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)


def guardar(fig, nombre):
    fig.savefig(FIG_DIR / f"03_{nombre}.png", bbox_inches="tight")

In [ ]:
train, _ = paths.load_data()
X = paths.texts(train)
y = train["age_range"].to_numpy()

modelos = {s: EA.cargar(X, y, s, "greedy_ent", N_JOBS) for s in (AJ, CF)}
m = modelos[CF]  # semilla de confirmación: la que se analiza en las secciones 1–5
preds = {"vigente": m.p_vig, "greedy_ent": m.p_greedy}
y_va = m.y_idx
pred_g = E.as_pred(m.p_greedy)
err_g = pred_g != y_va
print(f"Tweets de validación (confirmación): {len(y_va):,}")
EA.verificar(modelos).round(4)

## 1. Resultados por clase

### 1.1 Precision, recall y F1

**En palabras simples:** para cada rango de edad, qué tan bien le va al modelo. Las columnas `real` y `predichas` dicen cuántos tweets hay realmente de cada clase y cuántos el modelo *dijo* que eran de esa clase: si `predichas` es mucho mayor que `real`, el modelo abusa de esa clase (y su precision baja); si es menor, la subestima (y su recall baja).

In [ ]:
EA.reporte_clases(y_va, preds).round(3)

El mismo F1 por clase en las **dos** semillas, para comprobar que las diferencias no son casualidad de una partición:

In [ ]:
pd.DataFrame({
    (nombre, col): E.f1_per_class(mm.y_idx, p)
    for col, mm in (("ajuste", modelos[AJ]), ("confirmación", modelos[CF]))
    for nombre, p in (("vigente", mm.p_vig), ("greedy_ent", mm.p_greedy))
}).T.round(3)

**Qué vemos.**

- `greedy_ent` mejora el F1 de **las 6 clases en las dos semillas** (no es una mejora concentrada en una clase ni en una partición). En confirmación el F1 macro pasa de 0.587 a 0.594.
- Las clases de los extremos son las más fáciles (65-xx 0.71, 50-64 0.66, 13-17 0.65) y las **centrales las más difíciles** (25-34 0.48, 18-24 0.50): quedan "encerradas" entre dos vecinas parecidas.
- El desbalance de predicciones sigue: el modelo **dice 18-24 de más** (6 666 predicciones para 5 735 tweets reales, peor que la vigente) y **dice 35-49 y 13-17 de menos** (4 806 de 5 646 y 5 226 de 5 773). Por eso 18-24 tiene recall alto (0.54) pero precision baja (0.47): funciona como "clase cajón" para los tweets jóvenes dudosos. Corregirlo con sesgos por clase no funcionó en el 02 (sección 9).

### 1.2 Matriz de confusión

**En palabras simples:** cada fila es la edad real y cada columna la edad que dijo el modelo. Cada fila suma 1: por ejemplo, la celda (real 25-34, predicha 18-24) = 0.20 significaría que el 20 % de los tweets de 25-34 se clasificaron como 18-24. La diagonal son los aciertos (= recall). Si los errores se acumulan **pegados a la diagonal**, el modelo se equivoca "por poco" (con la clase vecina).

La segunda figura resta las dos matrices (greedy_ent − vigente), en puntos porcentuales: azul en la diagonal = greedy_ent acierta más en esa clase; azul fuera de la diagonal = comete más ese error concreto.

In [ ]:
cms = {n: EA.confusion(y_va, p) for n, p in preds.items()}
fig = EA.plot_confusiones(cms)
guardar(fig, "confusion")
plt.show()
fig = EA.plot_diferencia_confusion(cms["vigente"], cms["greedy_ent"], "vigente", "greedy_ent")
guardar(fig, "confusion_diferencia")
plt.show()

**Qué vemos.**

- Los errores se acumulan **pegados a la diagonal**. Los dos más grandes son 13-17 → 18-24 (22 % de los tweets de 13-17) y 25-34 → 18-24 (21 %); le siguen 18-24 → 13-17 (16 %) y 18-24 → 25-34 (15 %).
- **25-34 se escapa hacia abajo**: va 2 veces más a 18-24 (21 %) que a 35-49 (11 %). Sus tweets se parecen más a los de los jóvenes que a los de los adultos.
- **35-49 es la clase más "dispersa"**: sus errores se reparten casi por igual entre 18-24, 25-34, 50-64 y 65-xx (10–13 % cada uno); no tiene un estilo propio que la delimite.
- Entre las clases mayores (50-64 ↔ 65-xx) el error es simétrico, de ≈ 10 % en cada sentido.
- La diferencia entre modelos es pequeña y casi toda favorable en la diagonal: +1.6 puntos de recall en 18-24, +1.0 en 25-34, +1.4 en 35-49 y +0.6 en 65-xx; 13-17 y 50-64 pierden menos de un punto de recall, pero ganan precision (ver 1.1).

### 1.3 ¿Los errores caen en la clase vecina o lejos?

**En palabras simples:** los rangos de edad tienen un orden (13-17 < 18-24 < ... < 65-xx). Equivocarse de 25-34 a 18-24 (distancia 1) es un error "razonable"; equivocarse de 13-17 a 65-xx (distancia 5) es un error grave. La tabla dice qué % de los errores cae a cada distancia.

In [ ]:
EA.errores_ordinales(y_va, preds).round(2)

**Qué vemos.**

- El 57 % de los errores cae en la **clase vecina** y el 82 % a distancia ≤ 2. Solo el 18 % de los errores (≈ 7 % de los tweets) son "graves" (3 o más rangos de distancia).
- Esto confirma lo que mostró el 01: la señal de edad es **gradual** (el estilo cambia poco a poco con la edad), así que los límites entre rangos vecinos son borrosos por naturaleza. Parte de este error es irreducible con solo un tweet: una persona de 24 y otra de 25 años escriben igual.
- `greedy_ent` mueve un poco los errores hacia las vecinas (57.2 % frente a 56.5 %), pero la forma es casi la misma: los cambios del 02 mejoran el modelo de forma pareja, no cambian el tipo de error.

## 2. Errores por tipo de tweet

**En palabras simples:** se marca cada tweet de validación según su tipo (¿es un retweet?, ¿tiene un enlace?, ¿es muy corto?...) y se mide la accuracy dentro de cada grupo. Así se ve qué tweets son difíciles por naturaleza. Un tweet puede estar en varios grupos a la vez (un RT con URL cuenta en ambos).

Cómo leer las columnas:

- `% de los tweets`: qué tan común es ese tipo.
- `% de los errores (greedy_ent)`: qué parte de todos los errores del modelo cae en ese tipo. Si es mayor que `% de los tweets`, ese tipo concentra errores.
- `accuracy` y `F1` de cada modelo dentro del grupo, y `Δ accuracy` = cuánto mejoró greedy_ent en ese grupo.

| Tipo | Regla (la misma del `01_exploracion`) |
|---|---|
| sin texto propio | al quitar URLs, menciones, hashtags y "RT" no queda ninguna palabra |
| RT | empieza con `RT @usuario` |
| mención al inicio | empieza con `@usuario` (es una respuesta) |
| plantilla frecuente en train | su "esqueleto" (URL → `URL`, `@x` → `@U`, números → `0`) aparece ≥ 5 veces en el train de esta semilla |
| llega a 140 caracteres | probablemente cortado por el límite de Twitter |
| mayúsculas sostenidas | ≥ 70 % de las letras en mayúscula y al menos 10 letras |
| emoticono / risa / alargamiento | `:)`, `<3`... / `jaja`, `xd`... / `siii`, `nooo` |
| longitud | en caracteres |

In [ ]:
tipos = EA.tipos_de_tweet(m.X_va, m.X_tr)
long_va = EA.longitud(m.X_va)
por_tipo = EA.metricas_por_grupo(y_va, preds, EA.grupos(tipos, long_va))
por_tipo.round(3)

In [ ]:
fig = EA.plot_grupos(por_tipo)
guardar(fig, "accuracy_por_tipo")
plt.show()

**Qué vemos.**

- **Lo más difícil son los tweets con poco texto.** Los de 0–30 caracteres son el 9.9 % de los tweets pero el 13.4 % de los errores (accuracy 0.45). Los tweets sin texto propio (solo enlace, menciones o RT vacío) aciertan el 51 %, y los que siguen una plantilla frecuente (`URL`, `RT @U: URL`, "acabo de publicar una foto"...) el 57 %. En estos casos el modelo casi solo puede adivinar por a quién se menciona o por el enlace.
- **La accuracy sube con la longitud**: 0.45 (0–30) → 0.55 → 0.58 → 0.62 → 0.66 (121–139). Más texto = más evidencia. Los tweets truncados a 140 aciertan algo menos (0.64) que los de 121–139: el corte pierde el final del mensaje.
- **Lo más fácil son las respuestas (mención al inicio, 0.70) y los tweets con hashtag (0.69)**: ahí los canales de menciones y hashtags aportan una señal "social" fuerte (a quién se le habla, qué comunidad se sigue).
- **Dónde mejoró greedy_ent:** sobre todo en los tweets con **mayúsculas sostenidas** (+3.2 puntos de accuracy): es justo lo que recuperan los canales `char*_cs` sin casefold. También en tweets con hashtag, alargamientos y de 31–60 caracteres (≈ +1 punto). **Dónde no mejoró:** tweets sin texto propio, plantillas y emoticonos (≈ 0): ningún cambio del 02 agrega información para ellos.

## 3. Confianza y calibración

### 3.1 ¿Acierta más cuando está seguro?

**En palabras simples:** el **margen** mide qué tan "convencido" está el modelo (probabilidad de su 1.ª opción menos la de su 2.ª). Se ordenan los tweets de menor a mayor margen y se parten en 5 grupos iguales (quintiles, Q1 = el 20 % más dudoso). Si el modelo es útil, la accuracy debe subir de Q1 a Q5. La columna `% de los errores` muestra dónde se concentran los fallos.

In [ ]:
conf = {n: EA.tabla_confianza(y_va, p) for n, p in preds.items()}
display(conf["greedy_ent"].round(3))
fig = EA.plot_confianza(conf)
guardar(fig, "confianza")
plt.show()

**Qué vemos.**

- El margen es un muy buen indicador de cuándo creerle al modelo: en el 20 % más seguro (Q5, margen > 0.53) **acierta el 96 %**; en el 20 % más dudoso (Q1, margen < 0.07, prácticamente un empate) acierta solo el 32 %.
- **El 64 % de todos los errores está en los dos quintiles más dudosos** (Q1 + Q2). La mayoría de los errores no son el modelo "seguro y equivocado", sino el modelo dudando entre dos o tres clases (normalmente vecinas) y eligiendo mal.
- Consecuencia práctica: mejorar el modelo es sobre todo **desempatar** los casos dudosos; los casos claros ya se resuelven bien.

### 3.2 Calibración

**En palabras simples:** si el modelo dice "estoy 70 % seguro", ¿acierta el 70 % de esas veces? Se agrupan los tweets por la probabilidad que el modelo le dio a su clase elegida y se compara con la accuracy real del grupo. En el gráfico, la diagonal gris es la calibración perfecta: por **debajo** de la diagonal el modelo es *demasiado confiado*; por **encima**, *demasiado modesto*. El **ECE** resume la distancia promedio a la diagonal (0 = perfecto).

Para el F1 solo importa *cuál* clase gana, no el valor exacto de la probabilidad, así que una mala calibración no baja el F1; pero importa si se quisieran usar las probabilidades (por ejemplo, para decidir cuándo confiar en la predicción).

In [ ]:
cal = {n: EA.calibracion(y_va, p) for n, p in preds.items()}
display(cal["greedy_ent"][0].round(3))
fig = EA.plot_calibracion(cal)
guardar(fig, "calibracion")
plt.show()

**Qué vemos.**

- La mezcla es **demasiado modesta** en la parte alta: cuando dice 65 % acierta el 88 %, y cuando dice 75 % acierta el 95 %. Solo en la zona de duda (confianza 0.2–0.4) la probabilidad coincide más o menos con la accuracy.
- La causa es la propia mezcla: promediar las probabilidades de 12 canales "aplana" el resultado (un canal muy seguro queda diluido por los demás). El ECE es 0.115 en greedy_ent y 0.126 en la vigente.
- No afecta el F1 (solo importa qué clase gana), pero significa que las probabilidades **no se pueden leer al pie de la letra**. Si se quisieran usar (por ejemplo, para decidir cuándo confiar en la predicción), habría que recalibrarlas; para eso sirve el margen de 3.1, que sí ordena bien los casos.

## 4. Qué cambió entre la mezcla vigente y `greedy_ent`

**En palabras simples:** se comparan los dos modelos tweet por tweet:

- **corregida**: la vigente fallaba y greedy_ent acierta.
- **estropeada**: la vigente acertaba y greedy_ent falla.
- **ambos fallan**: con la misma clase o cambiando de clase equivocada.

La mejora neta es *corregidas − estropeadas*. Un modelo mejor no solo arregla tweets: también daña algunos; lo que importa es el saldo.

In [ ]:
cambio = EA.cambios(y_va, m.p_vig, m.p_greedy)
resumen_cambio = cambio.value_counts().reindex(EA.CATEGORIAS_CAMBIO).to_frame("tweets")
resumen_cambio["%"] = (100 * resumen_cambio["tweets"] / len(cambio)).round(2)
resumen_cambio

**Por clase real** (`neto por cada 1000 tweets` permite comparar clases de distinto tamaño):

In [ ]:
EA.tabla_cambios(cambio, EA.grupos_clase(y_va)).round(1)

**Por tipo de tweet:**

In [ ]:
EA.tabla_cambios(cambio, EA.grupos(tipos, long_va)).round(1)

**Ejemplos** (sorteados con semilla fija). Las dos últimas columnas dicen qué canales de greedy_ent empujaron hacia la clase correcta y cuáles hacia la rival (ver sección 5 para cómo se calcula).

In [ ]:
votos, rival = EA.votos_canales(m)
print("Corregidas")
display(EA.tabla_ejemplos(m, EA.muestrear(cambio == "corregida", 5, SEMILLA_EJEMPLOS), votos, rival))
print("Estropeadas")
display(EA.tabla_ejemplos(m, EA.muestrear(cambio == "estropeada", 5, SEMILLA_EJEMPLOS), votos, rival))

**Qué vemos.**

- Entre los dos modelos cambia la predicción del **10.9 % de los tweets**, pero el saldo es pequeño: 1 182 corregidas frente a 985 estropeadas, **neto +197 tweets (0.6 %)**. La mejora del F1 viene de muchos cambios en ambos sentidos que, sumados, salen a favor.
- **Por clase**, el saldo es positivo en 18-24 (+91), 35-49 (+80), 25-34 (+55) y 65-xx (+37), y negativo en 13-17 (−44) y 50-64 (−22). Aun así el F1 de 13-17 y 50-64 sube, porque greedy_ent les asigna menos tweets de otras clases por error (sube su precision).
- **Por tipo**, la mejora neta más alta está en los tweets con mayúsculas sostenidas (+32 por cada 1 000), lo que coincide con la sección 2. En tweets sin texto propio, plantillas y emoticonos el saldo es ≈ 0.
- Los ejemplos de estropeadas son casi siempre **empates** (por ejemplo 0.42 / 0.42 entre 13-17 y 18-24): un cambio pequeño en los pesos los inclina hacia un lado u otro. Por eso las corregidas y estropeadas se parecen tanto.

## 5. Ejemplos de errores típicos y qué canal "votó" mal

### 5.1 Cómo se mide el voto de cada canal

**En palabras simples:** para cada tweet se toma la clase **real** y la **rival** (la que el modelo eligió si se equivocó, o su 2.ª opción si acertó). El **margen** es probabilidad(real) − probabilidad(rival): positivo = acierta, negativo = falla. Luego se "apaga" un canal (se le hace votar lo mismo para las 6 clases, como si se abstuviera) y se vuelve a medir el margen. La diferencia es el **voto** de ese canal:

- **voto > 0**: el canal empujó hacia la respuesta correcta.
- **voto < 0**: el canal empujó hacia el error.

Los canales con más peso en la mezcla (por ejemplo `char6_cs`, que lleva 40 % del bloque de caracteres) votan más fuerte, tanto para bien como para mal; por eso conviene mirar a la vez el voto medio en aciertos y en errores.

In [ ]:
EA.resumen_votos(votos, err_g).round(4)

**¿Cambia el culpable si el tweet es un retweet?** % de los errores en que cada canal fue el que más empujó hacia el error, separando RT y no RT.

In [ ]:
EA.culpables(votos, err_g, tipos["RT"].to_numpy(), ("no es RT", "es RT")).round(1)

**Qué vemos.**

- Todos los canales votan, en promedio, a favor en los aciertos y en contra en los errores; ninguno es "malo". Los errores ocurren cuando la evidencia del tweet apunta de verdad a otra clase.
- Los que más veces son el principal culpable son los de **más peso**: `char6_cs` (30 % de los errores), `word_seq3_ent5_tt` (25 %) y `char_seq4` (14 %). No es que sean peores: votan más fuerte, para bien y para mal.
- `mention_pos_tt` y `mentions` son un caso distinto: tienen el voto medio más alto en los aciertos (0.07 y 0.05) y solo empujan al error en el 37 % y 61 % de los errores, pero cuando se equivocan se equivocan fuerte (−0.15 en varios ejemplos).
- **En los retweets, los canales de menciones pasan a ser culpables en el 35 % de los errores** (frente al 10 % en los no RT). En un RT, `@cuenta` es el autor del tweet original, no a quien se le habla: el canal aprende "quién retuitea a esa cuenta", que es una señal de comunidad, no de la edad del autor (ver los ejemplos de `@SamsungCO`, `@MissPiggy` o `@McDonaldsCol` en 5.3).

### 5.2 Voto medio de cada canal en las confusiones más frecuentes

Cada columna es un tipo de error (real → predicha) y `n` cuántos tweets de validación lo tienen. Valores negativos grandes = ese canal es el que más arrastra hacia ese error.

In [ ]:
vpp = EA.votos_por_par(votos, y_va, m.p_greedy)
vpp.style.format("{:+.3f}").background_gradient(cmap="RdBu", vmin=-0.05, vmax=0.05, axis=None)

**Qué vemos.**

- **Entre las clases jóvenes** (13-17 ↔ 18-24 ↔ 25-34) los errores los empujan sobre todo los canales de **estilo** (`char6_cs`, `char_seq4`, `word_seq3_ent5_tt`): la forma de escribir de un adolescente y de un universitario se parece.
- **Entre las clases mayores** (35-49 ↔ 50-64 ↔ 65-xx) los canales **sociales** (`mentions`, `mention_pos_tt`) empujan al error 2–4 veces más que entre las jóvenes (≈ −0.03 frente a ≈ −0.01): los adultos mayores siguen y retuitean las mismas cuentas de política y noticias, así que la mención no permite separar 50-64 de 65-xx.
- `shape7` es el único canal que en algunos errores de jóvenes empuja hacia la respuesta correcta en promedio (18-24 → 25-34 y 18-24 → 13-17): la forma del texto (mayúsculas, puntuación) aporta una señal que los demás canales no tienen.

### 5.3 Ejemplos reales

Seis tweets por tipo de error, sorteados con semilla fija. En `top-3 greedy_ent` se ven las tres clases más probables: muchos errores son "empates" entre clases vecinas.

In [ ]:
for real, pred in [("18-24", "25-34"), ("25-34", "18-24"), ("13-17", "18-24"), ("18-24", "13-17"),
                   ("65-xx", "50-64"), ("50-64", "65-xx")]:
    mask = EA.mascara_par(y_va, m.p_greedy, real, pred)
    print(f"\nReal {real} → predicha {pred}  ({mask.sum()} tweets en validación)")
    display(EA.tabla_ejemplos(m, EA.muestrear(mask, 6, SEMILLA_EJEMPLOS), votos, rival))

**Errores lejanos** (distancia ≥ 3 rangos: por ejemplo un adolescente clasificado como 50-64):

In [ ]:
lejos = np.abs(pred_g - y_va) >= 3
print(f"Errores a distancia ≥ 3: {lejos.sum():,} ({lejos.mean():.1%} de los tweets)")
EA.tabla_ejemplos(m, EA.muestrear(lejos, 8, SEMILLA_EJEMPLOS), votos, rival)

In [ ]:
# ¿Qué tipo de tweet son los errores lejanos? (% dentro de los errores lejanos frente al total)
pd.DataFrame({"% en errores lejanos": 100 * tipos[lejos].mean(),
              "% en todos los tweets": 100 * tipos.mean()}).round(1)

**Qué vemos en los ejemplos.**

- **Empates entre vecinas.** Muchos errores tienen la 1.ª y 2.ª clase a pocos puntos ("18-24 0.29, 25-34 0.27"). Son tweets genéricos ("Si no has amanecido con tus amigos riendote...") que podría haber escrito alguien de cualquiera de las dos edades.
- **Retweets: el texto no es del autor.** Varios errores son RT de cuentas de humor, marcas o política ("RT @DepressedDarth: Friend-zone", "RT @luisvicenteleon: El resto del decreto..."). La etiqueta es la edad de quien retuitea, pero el texto lo escribió otra persona, y el modelo clasifica el estilo del autor original.
- **Sin texto.** Tweets que son solo un enlace o solo una mención ("@ozambranoo", "https://t.co/...") o vacíos: el modelo no tiene nada de qué agarrarse y vota por la clase con más evidencia previa.
- **Idioma.** Tweets en inglés ("@loquaciousfool yup, by gerald durrell! great book") se asocian a 13-17, donde el inglés es frecuente por el fandom de música.
- **Errores lejanos.** Los errores a 3 o más rangos (7.2 % de los tweets) son, de forma desproporcionada, tweets **sin texto propio** (13.3 % de los errores lejanos frente a 5.9 % del total), con **plantilla frecuente** (14.1 % frente a 6.7 %) y **con URL** (49.8 % frente a 40.5 %); y tienen menos respuestas, emoticonos y risas, que son las señales más claras de edad.

## 6. Ablación: cuánto aporta cada pieza

**En palabras simples:** se quita una pieza de `greedy_ent` a la vez y se mide cuánto baja el F1. Lo que baja es lo que esa pieza aportaba (`aporte`). Se hace en las **dos** semillas para no confiar en una sola partición, y en la de confirmación se calcula un intervalo de confianza al 95 % (bootstrap: se re-sortean los tweets 1 000 veces).

Hay dos tipos de filas:

- **quitar canal**: el canal se apaga (vota igual para todas las clases) y los demás quedan con sus pesos.
- **revertir**: se deshace una decisión del camino greedy del 02 (volver a minúsculas, quitar las entidades, volver al tokenizador regex, quitar un canal agregado). La última fila deshace todo y es la mezcla vigente.

`ayuda en ambas` = aporte positivo en las dos semillas e intervalo por encima de 0.

In [ ]:
abl = EA.ablacion(modelos)
abl.sort_values("aporte confirmacion", ascending=False).round(4)

In [ ]:
fig = EA.plot_ablacion(abl)
guardar(fig, "ablacion")
plt.show()

**Qué vemos.**

- **Lo que más aporta es la información social y la de palabras:** apagar `word_seq3_ent5_tt` cuesta −0.0064 de F1 en confirmación, `mention_pos_tt` −0.0048 y `mentions` −0.0034; los tres con intervalo de confianza por encima de 0 y en ambas semillas.
- **De las decisiones del camino greedy**, las que más valen son las **mayúsculas** (volver a `char*` con casefold cuesta −0.0046) y las **entidades** (volver a `word_seq3` cuesta −0.0017). Ambas pasan la regla en las dos semillas. Esto coincide con la sección 2 (gana en tweets con mayúsculas) y con la 5 (la señal social pesa).
- **`char_seq4` y `shape7` aportan poco y no es concluyente** (≈ +0.001, con intervalo que incluye 0). En el 02 se aceptaron mirando ambas semillas, así que su aporte estaba algo sobreestimado (ya estaba anotado como limitación).
- **TweetTokenizer no aporta por sí solo** una vez están las entidades y las mayúsculas (−0.0001): su ganancia en el 02 se solapa con la de esos cambios.
- **Los tres canales de caracteres son redundantes entre sí:** quitar `char5_cs` o `char6_cs` por separado incluso sube un poco el F1 (+0.0009), porque su peso se reparte entre los otros dos, que dicen casi lo mismo. Pero quitar las mayúsculas de los tres a la vez sí duele (−0.0046): lo que importa es que **algún** canal de caracteres vea las mayúsculas.
- Nota: "quitar canal" y "revertir: sin X (agregado)" dan casi lo mismo para `shape7` y `char_seq4`; la pequeña diferencia es que quitar el agregado también cambia el peso relativo del agregado que va después.

## 7. Limitaciones y conclusiones

### Limitaciones

**De los datos (no se arreglan con un mejor modelo):**

1. **Un tweet tiene muy poca información.** El 9.9 % de los tweets de validación tiene 30 caracteres o menos y el 5.9 % no tiene texto propio; ahí la accuracy cae a 0.45–0.51. Con un solo tweet por persona no hay forma de acumular evidencia del autor.
2. **La etiqueta es del autor, pero el texto muchas veces no.** El 36 % de los tweets son retweets: el texto lo escribió otra persona y el modelo aprende el estilo y la comunidad de esa otra cuenta. En los errores de RT, los canales de menciones son el principal culpable en el 35 % de los casos.
3. **La edad es continua y los rangos son cortes artificiales.** El 57 % de los errores va a la clase vecina y el 64 % de los errores son casos donde el modelo dudaba (margen < 0.16). Entre 24 y 25 años no hay un cambio de estilo que se pueda detectar.
4. **Plantillas y bots.** Los mensajes automáticos ("acabo de publicar una foto", contadores de seguidores, solo un enlace) aparecen con etiquetas de todas las clases; su accuracy es 0.57 y ningún cambio del 02 los mejoró.
5. **El texto llega normalizado a ASCII** (sin tildes, ñ ni emojis) y un 2.5 % está truncado a 140 caracteres: se perdieron señales de estilo (emojis, ortografía) que probablemente ayudarían a separar edades.

**Del modelo:**

6. **Clases centrales.** 25-34 (F1 0.48) y 18-24 (0.50) siguen siendo las más débiles: 25-34 no tiene vocabulario propio (01) y el 21 % de sus tweets se va a 18-24. 18-24 funciona como clase "cajón" y se sobre-predice (6 666 predicciones para 5 735 tweets).
7. **Probabilidades mal calibradas.** La mezcla es demasiado modesta (ECE 0.115): cuando dice 65 % acierta el 88 %. No afecta el F1, pero impide usar las probabilidades directamente.
8. **Señal social con doble filo.** Menciones y hashtags son de lo que más aporta (ablación), pero dependen de que la cuenta ya aparezca en train (el 75.7 % de las menciones de test sí aparece) y confunden a las clases mayores, que siguen las mismas cuentas de política.
9. **Canales redundantes.** `char4_cs`, `char5_cs` y `char6_cs` dicen casi lo mismo; `char_seq4`, `shape7` y TweetTokenizer no tienen un aporte concluyente una vez está el resto.

**De la evaluación:**

10. Se usaron dos particiones de un fold (≈ 20 %) cada una, no validación cruzada completa. La semilla de confirmación también se miró para aceptar `shape7` y `char_seq4` (02, sección 13), así que las métricas de `greedy_ent` en esa semilla son algo optimistas; la ablación lo confirma (su aporte no es concluyente).

### Conclusiones

1. `greedy_ent` mejora a la mezcla vigente en las 6 clases y en las dos semillas (F1 0.587 → 0.594 en confirmación), sin cambiar el tipo de error: es una mejora pareja.
2. Casi todo el error es **entre clases vecinas y en casos dudosos**: el 57 % va a la vecina y el 64 % ocurre con margen bajo. Cuando el modelo está seguro (20 % de los tweets) acierta el 96 %.
3. Los errores graves se concentran en tweets **sin texto, con plantilla o solo con URL**, y en **retweets** (el texto no es del autor). Son límites de los datos más que del modelo.
4. Lo que más aporta al modelo es la **información social** (a quién se menciona, qué cuentas frecuentes aparecen) y recuperar las **mayúsculas**. Los demás cambios del camino greedy aportan poco por separado.
5. Para ir más allá con modelos de n-gramas habría que atacar los casos de poco texto y los RT (por ejemplo, separar el texto retuiteado del aporte del autor), más que agregar más canales de caracteres, que ya son redundantes entre sí.